In [ ]:
# Phase 1 smoke test: validates the offline import + decode + submission round-trip
# on real Kaggle hardware. Model is UNTRAINED (random init) -- the score is
# meaningless. What this proves: (a) the private-dataset package import works with
# internet off, (b) real per-slice decode timing on Kaggle's storage/CPU, (c) the
# submission.csv header and shape are correct end to end against the real test set.
import glob, os, shutil, sys, time

print('contents of /kaggle/input:', os.listdir('/kaggle/input'))
for d in os.listdir('/kaggle/input'):
    full = os.path.join('/kaggle/input', d)
    if os.path.isdir(full):
        print(f'  {d}/:', os.listdir(full))

# newer Kaggle input layout namespaces by source type, not a flat /kaggle/input/<slug>
src_candidates = glob.glob('/kaggle/input/**/rsna-knee-src', recursive=True)
comp_candidates = glob.glob('/kaggle/input/**/rsna-knee-abnormality-detection', recursive=True)
print('src_candidates:', src_candidates)
print('comp_candidates:', comp_candidates)
SRC = src_candidates[0]
COMP_DIR = comp_candidates[0]

PKG = '/kaggle/working/knee'
os.makedirs(PKG, exist_ok=True)
for fname in os.listdir(SRC):
    if fname.endswith('.py'):
        shutil.copy(os.path.join(SRC, fname), os.path.join(PKG, fname))
sys.path.insert(0, '/kaggle/working')

import knee.dicom as knee_dicom
import knee.infer as knee_infer
import knee.dataset as knee_dataset
import knee.model as knee_model
print('knee package imported successfully from', PKG)

In [ ]:
import pandas as pd

test_df = pd.read_csv(f'{COMP_DIR}/test.csv')
test_series_df = pd.read_csv(f'{COMP_DIR}/test_series.csv')
print(f'{len(test_df)} test studies, {len(test_series_df)} test series rows')

In [ ]:
import torch

# pretrained=False: internet is off for the submitted notebook, so ImageNet
# weights can't be downloaded. This model is untrained either way for this
# smoke test -- only the round-trip is being validated here, not accuracy.
model = knee_model.KneeModel(backbone_name='efficientnet_b0', num_labels=12, pretrained=False)
model.eval()

# Labels with no lexical rule yet (see knee.reports) get no real training
# signal in Phase 1 -- clamp those heads to the 0.5 benchmark floor rather
# than emit a random-init model's arbitrary output for them.
SUPPORTED_LABELS = {'Effusion', "Baker's", 'ACL', 'Medial Meniscus'}
UNSUPPORTED_IDX = [i for i, l in enumerate(knee_infer.LABEL_COLUMNS) if l not in SUPPORTED_LABELS]
print('unsupported (clamped to 0.5) label indices:', UNSUPPORTED_IDX)

In [ ]:
import numpy as np

dataset = knee_dataset.KneeStudyDataset(
    study_uids=test_df['StudyInstanceUID'].tolist(),
    dcm_root=f'{COMP_DIR}/test_series',
    series_df=test_series_df,
    n_slices=16,
    size=224,
    max_series=1,
)

per_study_seconds = []

def predict_one(study_uid):
    idx = dataset.study_uids.index(study_uid)
    t0 = time.time()
    image, _, _ = dataset[idx]
    with torch.no_grad():
        logits = model(image.unsqueeze(0))
        probs = torch.sigmoid(logits)[0].numpy()
    probs[UNSUPPORTED_IDX] = 0.5
    per_study_seconds.append(time.time() - t0)
    return probs

wall_start = time.time()
submission = knee_infer.build_submission(test_df['StudyInstanceUID'].tolist(), predict_one)
wall_total = time.time() - wall_start

print(f'total wall time for {len(test_df)} studies: {wall_total:.2f}s')
if per_study_seconds:
    print(f'mean per-study time: {np.mean(per_study_seconds):.3f}s')
    print(f'extrapolated to 1300 studies: {np.mean(per_study_seconds) * 1300 / 60:.1f} min')

In [ ]:
submission.to_csv('submission.csv', index=False)
print(submission)